# Portfolio Lab · LQ45 (riset, bukan rekomendasi)
Notebook ini memakai rumus yang sama dengan web IDX Evidence Lab. Tidak mengambil API, tidak mengeksekusi transaksi, dan tidak menyajikan tanggal pasti drawdown masa depan. Jalankan sel berurutan setelah memasang paket data lokal.
Roadmap: bundel + hash -> inspeksi sumber -> tanggal bersama -> EDA -> baseline -> BL/Markowitz/diversifikasi -> CAPM dan risiko -> walk-forward -> skenario -> kesimpulan dan manifest. Dependensi: numpy, scipy, matplotlib; gunakan requirements-portfolio.txt di bundel. Notebook disimpan tanpa output; eksekusi berada di bawah kontrol pengguna.


## Sumber dan provenance
Di laptop proyek jalankan `python3 scripts/export_portfolio_lab_notebook_data.py` untuk membuat `data/processed/portfolio_lab_bundle.zip` dari snapshot Sectors.app yang sudah tersimpan. Di Colab, unggah ZIP itu dengan panel Files ke folder kerja; di Kaggle, attach ZIP sebagai Dataset dan ubah `BUNDLE_PATH` ke path file ZIP di `/kaggle/input/...`. Tidak perlu kunci API. Jika belum ada ZIP, sel berikut berhenti dengan pesan yang dapat ditindaklanjuti. Snapshot universe memiliki masalah checksum metadata; loader mengungkapnya dan memeriksa keanggotaan tiap emiten lewat laporan perusahaan, bukan mengklaim universe terverifikasi penuh.


In [ ]:
from pathlib import Path
import hashlib, json, sys, zipfile
BUNDLE_PATH = Path('portfolio_lab_bundle.zip')  # Colab: unggah ke Files; Kaggle: isi path ZIP pada Dataset terpasang
WORK_ROOT = Path('portfolio_lab_offline').resolve()
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f'Bundel tidak ditemukan: {BUNDLE_PATH}. Ekspor dari proyek lalu unggah/attach ZIP ke sesi ini.')
with zipfile.ZipFile(BUNDLE_PATH) as bundle:
    names = bundle.namelist()
    if any(name.startswith('/') or '..' in Path(name).parts for name in names):
        raise ValueError('Path ZIP tidak aman')
    manifest = json.loads(bundle.read('manifest.json'))
    for name, expected in manifest['source_files'].items():
        if hashlib.sha256(bundle.read(name)).hexdigest() != expected:
            raise ValueError(f'Hash sumber tidak cocok: {name}')
    bundle.extractall(WORK_ROOT)
sys.path.insert(0, str(WORK_ROOT / 'src'))
print('Bundel tervalidasi:', manifest['provider'], manifest['ticker_count'], 'emiten;', manifest['file_count'], 'berkas')


## Penyiapan data
Pilih 3–10 ticker dari universe saat ini. Loader menyelaraskan hanya tanggal bersama dengan benchmark, tidak mengisi tanggal kosong, dan mengecualikan lompatan harga >40% yang belum dijelaskan. Return adalah *price return* provisional, bukan total return yang sudah disesuaikan dividen/corporate action.


In [ ]:
from idx_evidence_lab.portfolio_data import load_portfolio_inputs
TICKERS = ['BBCA', 'BBRI', 'BMRI', 'TLKM', 'ASII']
BENCHMARK = 'IHSG'
inputs = load_portfolio_inputs(WORK_ROOT, TICKERS, BENCHMARK)
print('Status:', inputs['status'], '| Kualitas:', inputs['quality'])
print('Cakupan:', inputs['coverage'])
print('Isu sumber:', inputs['issues'][:8])
from idx_evidence_lab.market_data import load_issuer_daily, load_index_daily_snapshot
for ticker in TICKERS:
    snapshot = load_issuer_daily(WORK_ROOT, ticker)
    print(ticker, snapshot['data_quality'], snapshot['observation_count'], snapshot['coverage_start'], snapshot['coverage_end'], 'source:', inputs['sources']['daily'][ticker][:1], 'sample:', snapshot['series'][:1])
index_snapshot = load_index_daily_snapshot(WORK_ROOT, BENCHMARK)
print(BENCHMARK, index_snapshot['data_quality'], index_snapshot['observation_count'], index_snapshot['coverage_start'], index_snapshot['coverage_end'])
assert len(inputs['return_dates']) == len(set(inputs['return_dates']))
assert inputs['return_dates'] == sorted(inputs['return_dates'])
print('Audit: tanggal unik, terurut; loader memeriksa metadata/hash, missing, gap dan jump; sumber corporate action belum adjusted.')
if inputs['status'] != 'READY':
    raise RuntimeError('Riwayat/provenance belum memenuhi syarat; periksa isu di atas dan pilih emiten lain.')
DATES, RETURNS = inputs['return_dates'], inputs['return_matrix']


## EDA
Lihat pergerakan indeks harga yang dinormalisasi dan distribusi return harian. Grafik ini observasi historis, bukan prakiraan.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.asarray(RETURNS, dtype=float)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i, ticker in enumerate(TICKERS):
    axes[0].plot(DATES, np.cumprod(1 + r[:, i]), label=ticker, linewidth=1)
axes[0].set_title('Harga relatif · tanggal bersama'); axes[0].tick_params(axis='x', labelrotation=45); axes[0].legend()
axes[1].boxplot([r[:, i] * 100 for i in range(len(TICKERS))], tick_labels=TICKERS)
axes[1].set_title('Return harian (%) · outlier terlihat'); plt.tight_layout(); plt.show()


## Baseline
Equal-weight adalah pembanding sederhana; bobotnya tidak disembunyikan dan tidak diklaim optimal. Estimasi berikut memakai 252 sesi bersama terakhir jika tersedia.


In [ ]:
from idx_evidence_lab.portfolio_optimization import equal_weight_portfolio, optimize_portfolio
LOOKBACK = min(252, len(RETURNS))
training = RETURNS[-LOOKBACK:]
baseline = equal_weight_portfolio(training, TICKERS)
print('Bobot baseline:', baseline['weights'])
print('Volatilitas estimasi tahunan:', baseline['estimated_volatility_annual'])


## Optimasi
Alur utama memakai Black–Litterman untuk estimasi return, kemudian Markowitz dengan batas risiko dan minimum rasio diversifikasi. Prior pi = 2.5 Sigma w_cap pada kapitalisasi universe terpilih; P = I, Q = rerata excess return sampel, tau = 1/T, Omega = diag(Sigma)/T. Ini views statistik historis dan prior proxy, bukan views investor atau equilibrium seluruh pasar. Parameter tetap untuk eksperimen, belum terbukti lebih unggul. Maximum diversification dicari pada kendala risiko yang sama; floor = DR_minvar + f × (DR_maxdiv − DR_minvar), f = 0.75/0.50/0.25 untuk konservatif/moderat/agresif. HRP tetap pembanding internal. Factor Zoo merupakan karakteristik; faktor return Fama–French belum tersedia.


In [ ]:
PROFILE = 'moderate'  # conservative / moderate / aggressive
METHOD = 'integrated'
RISK_FREE_ANNUAL = 0.07129  # proxy ID10Y manual pengguna, konstan selama sampel
caps_by_date = dict(zip(inputs['dates'], inputs['market_cap_matrix']))
constraints = {'sectors': inputs['sectors'], 'prior_weights': caps_by_date[DATES[-1]], 'risk_free_annual': RISK_FREE_ANNUAL}
results = {(method, profile): optimize_portfolio(training, TICKERS, method, profile, constraints)
           for profile in ('conservative', 'moderate', 'aggressive')
           for method in ('integrated', 'markowitz', 'maximum_diversification', 'hrp')}
for (method, profile), item in results.items():
    print(method, profile, item['status'], item['estimated_volatility_annual'], item['diversification_ratio'])
selected = results[METHOD, PROFILE]
if selected['weights'] is None:
    raise RuntimeError(f'Pilihan tidak feasible: {selected["reason"]}')
weights = np.asarray([selected['weights'][ticker] for ticker in TICKERS])
plt.bar(TICKERS, weights * 100); plt.ylabel('Bobot (%)'); plt.title(f'{METHOD} · {PROFILE}'); plt.show()


## Metrik historis
Dua CAPM ditampilkan terpisah. CAPM historis memakai return IHSG aktual yang sejajar: Rm = 252 × mean(return IHSG harian), premi = Rm − RF; ini menjelaskan hurdle dan alpha historis in-sample. CAPM ATH memakai asumsi pemulihan ke ATH 9.174: return kumulatif (ATH / penutupan lokal terakhir − 1). Horizon tidak diketahui; upside kumulatif dipakai langsung sebagai Rm proxy tanpa annualisasi, sehingga hasilnya ambang heuristik dan bukan forecast return tahunan yang setara horizon. Keduanya memakai beta historis dan RF manual yang sama. Sharpe memakai excess return harian efektif. Metrik in-sample belum memperhitungkan biaya, dividen dan slippage.


In [ ]:
from idx_evidence_lab.portfolio_analytics import calculate_portfolio_metrics
ATH_IHSG = 9174.0  # level ATH diberikan pengguna; bukan estimasi forecast
market_return_assumption_annual = (ATH_IHSG / index_snapshot['latest_price'] - 1) if BENCHMARK == 'IHSG' and index_snapshot['data_quality'] == 'VERIFIED' else None
print('Asumsi return pasar berbasis ATH:', market_return_assumption_annual, '| harga IHSG lokal as-of:', index_snapshot['coverage_end'], '| level:', index_snapshot['latest_price'])
sample_dates = DATES[-LOOKBACK:]
portfolio_returns = (np.asarray(training) @ weights).tolist()
benchmark_returns = [inputs['benchmark_returns'][day] for day in sample_dates]
metrics = calculate_portfolio_metrics(sample_dates, portfolio_returns, benchmark_returns,
                                      risk_free_annual=RISK_FREE_ANNUAL, market_return_assumption_annual=market_return_assumption_annual)
historical_metrics = calculate_portfolio_metrics(sample_dates, portfolio_returns, benchmark_returns, risk_free_annual=RISK_FREE_ANNUAL)
print({key: metrics[key] for key in ('sharpe','sortino','calmar','beta','capm_hurdle')})
print('CAPM ATH:', metrics['capm_hurdle'], '| historis:', historical_metrics['capm_hurdle'], '| alpha historis:', historical_metrics['capm_alpha'])
print('Asumsi ATH:', metrics['assumptions']); print('Asumsi historis:', historical_metrics['assumptions'])
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].bar(['Rf manual', 'Beta × premi ATH', 'CAPM ATH'], [RISK_FREE_ANNUAL*100, metrics['beta']['value']*metrics['assumptions']['market_premium_annual']*100, metrics['capm_hurdle']['value']*100]); axes[0].axhline(0, color='gray'); axes[0].set_ylabel('%'); axes[0].set_title('Ambang heuristik ATH')
axes[1].bar(['Rf manual', 'Beta × premi historis', 'CAPM historis'], [RISK_FREE_ANNUAL*100, historical_metrics['beta']['value']*historical_metrics['assumptions']['market_premium_annual']*100, historical_metrics['capm_hurdle']['value']*100]); axes[1].axhline(0, color='gray'); axes[1].set_ylabel('% p.a.'); axes[1].set_title('CAPM dari IHSG historis')
axes[2].plot(sample_dates, [row['drawdown']*100 for row in metrics['path']]); axes[2].set_ylabel('Drawdown (%)'); axes[2].set_title('Maximum drawdown historis'); axes[2].tick_params(axis='x', labelrotation=45); plt.tight_layout(); plt.show()
from idx_evidence_lab.portfolio_amounts import allocate_lots
lot_result = allocate_lots(selected['weights'], 10000000, dict(zip(TICKERS, inputs['close_matrix'][-1])))
print('Ilustrasi modal Rp10 juta, lot 100 saham, tanpa biaya:', lot_result)


## Walk-forward
Bobot pada awal bulan dihitung hanya dari sesi sebelum tanggal penerapan, lalu dibandingkan dengan baseline equal-weight buy-and-hold pada tanggal holdout yang sama. Hasil *out-of-sample replay* tetap bruto sebelum biaya transaksi dan belum otomatis mengkalibrasi probabilitas skenario.


In [ ]:
from idx_evidence_lab.portfolio_scenarios import run_walk_forward, simulate_portfolio_scenarios
wf = run_walk_forward(DATES, RETURNS, TICKERS, METHOD, PROFILE, estimation_window=LOOKBACK, constraints=constraints, capitalization_by_date=caps_by_date)
print('Walk-forward:', wf['status'], 'holdout:', wf.get('holdout_sessions'))
if wf['status'] == 'READY':
    plt.plot([row['date'] for row in wf['path']], [row['wealth'] for row in wf['path']], label='Strategi')
    plt.plot([row['date'] for row in wf['baseline_path']], [row['wealth'] for row in wf['baseline_path']], label='Equal-weight')
    plt.legend(); plt.title('Replay historis · bruto'); plt.xticks(rotation=45); plt.show()


## Skenario
Multivariate circular block bootstrap memakai blok return lintas saham dan fixed-share buy-and-hold. P10/P50/P90 serta frekuensi melewati ambang drawdown adalah distribusi *eksploratoris*, bukan prediksi terkalibrasi atau tanggal pasti keluar/switch saham. Jumlah default 300 untuk notebook; web bisa memakai hingga 10.000 dengan batch memori.
MDD setiap lintasan adalah minimum dari nilai portofolio dibagi puncak sebelumnya dikurangi 1, termasuk modal awal 1. Batang p10 menunjukkan penurunan lebih dalam, p50 median, p90 lebih ringan; sekitar 10% lintasan berada sama atau lebih negatif dari p10, dan 90% sama atau lebih negatif dari p90. Nol hanya batas sumbu dan hasil yang sah untuk lintasan yang tidak pernah turun. MDD historis memakai periode sampel aktual, sedangkan ringkasan simulasi memakai 120 sesi; keduanya menjawab pertanyaan berbeda.


In [ ]:
SCENARIO_COUNT = 300
scenario = simulate_portfolio_scenarios(training, weights.tolist(), horizons=(20, 60, 120),
                                        simulations=SCENARIO_COUNT, block_size=5, seed=42,
                                        drawdown_threshold=-0.15, risk_free_annual=RISK_FREE_ANNUAL)
fan = scenario['fan']
sessions = [point['session'] for point in fan]
plt.fill_between(sessions, [p['p10'] * 100 for p in fan], [p['p90'] * 100 for p in fan], alpha=.25, label='P10–P90')
plt.plot(sessions, [p['p50'] * 100 for p in fan], label='Median skenario')
plt.xlabel('Sesi ke depan'); plt.ylabel('Return kumulatif (%)'); plt.legend(); plt.show()
print('Distribusi MDD:', scenario['horizons']['120']['max_drawdown'])
mdd_q = scenario['horizons']['120']['max_drawdown']
fig, ax = plt.subplots(figsize=(9, 3.4))
labels = ['p10 - lebih dalam', 'p50 - median', 'p90 - lebih ringan', 'MDD historis (periode sampel)']
values = [mdd_q['p10']*100, mdd_q['p50']*100, mdd_q['p90']*100, metrics['max_drawdown']['value']*100]
ax.barh(labels, values, color=['#e4777b', '#ffc86b', '#49c99e', '#65a8ff'], height=.45)
for i, value in enumerate(values): ax.text(value-.4, i, f'{value:.1f}%', ha='right', va='center')
ax.set_xlim(min(values)*1.3, 0); ax.axvline(0, color='gray', linewidth=1); ax.invert_yaxis()
ax.set_xlabel('Drawdown (%) - 0% adalah batas sumbu, bukan hasil MDD'); ax.set_title('MDD skenario 120 sesi dan pembanding historis')
ax.grid(axis='x', alpha=.2); plt.tight_layout(); plt.show()
print('Ambang -15%:', scenario['first_passage']['probability_by_horizon'])


## Kesimpulan
Trace hasil: bobot -> Markowitz dengan floor diversifikasi -> posterior Black–Litterman -> views rerata historis dan prior kapitalisasi -> tanggal bersama dan provenance Sectors. Evaluasi walk-forward memakai kapitalisasi sampai akhir training; RF konstan merupakan asumsi manual. Periksa kegagalan kendala tiap profil, sensitivitas delta/tau/floor pada validation, corporate action, biaya, serta universe historis sebelum menyimpulkan keunggulan. Tidak ada faktor return Fama–French atau APT yang diimputasi. Ekstensi data Sectors memerlukan approval terpisah.


## Manifest
Manifest ZIP merekam SHA-256 setiap input dan kode hitung. Jalankan sel berikut untuk melihat isi dan asumsi run; simpan bersama keluaran analisis.


In [ ]:
print(json.dumps({'provider': manifest['provider'], 'ticker_count': manifest['ticker_count'],
                  'file_count': manifest['file_count'], 'selected': TICKERS, 'benchmark': BENCHMARK,
                  'lookback': LOOKBACK, 'profile': PROFILE, 'method': METHOD,
                  'scenario_seed': scenario['seed'], 'scenario_count': SCENARIO_COUNT,
                  'risk_free_assumption': RISK_FREE_ANNUAL, 'limitations': manifest['limitations']},
                 indent=2, ensure_ascii=False))
